In [1]:
#load dataframes
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

#formatting 
BOLD = '\033[1m'
RESET = '\033[0m'

# Convenience functions, move to lib
def get_player_name_from_id(id):
    return competitors_df.loc[competitors_df['ID']==id,'Name'].values[0] 

# Load a local CSV file
competitors_df = pd.read_csv("data/competitors.csv")
rounds_df = pd.read_csv("data/rounds.csv")
submissions_df = pd.read_csv("data/submissions.csv")
votes_df = pd.read_csv("data/votes.csv")

# convenience definitions
playerIds = competitors_df['ID']

# submissions_df does not have a unique key, make one for easier lookups
submissions_df['uid'] = submissions_df['Spotify URI'] + ' ' + submissions_df['Round ID']
votes_df['uid'] = votes_df['Spotify URI'] + ' ' + votes_df['Round ID']

# Add a column in votes_df for submitted_by_id
lookup_series = submissions_df.set_index('uid')['Submitter ID']
votes_df['Submitter ID'] = votes_df['uid'].map(lookup_series)

# Print Datafram column headers
print('Competitors: ',competitors_df.columns.to_numpy().tolist())
print('Rounds: ',rounds_df.columns.to_numpy().tolist())
print('Submissions: ',submissions_df.columns.to_numpy().tolist())
print('Votes: ',votes_df.columns.to_numpy().tolist())

Competitors:  ['ID', 'Name']
Rounds:  ['ID', 'Created', 'Name', 'Description', 'Playlist URL']
Submissions:  ['Spotify URI', 'Title', 'Album', 'Artist(s)', 'Submitter ID', 'Created', 'Comment', 'Round ID', 'Visible To Voters', 'uid']
Votes:  ['Spotify URI', 'Voter ID', 'Created', 'Points Assigned', 'Comment', 'Round ID', 'uid', 'Submitter ID']


## Songs Submitted by Each User (interactive)

In [2]:
#list of songs submitted by each user
songs_submitted = {}
for player in playerIds:
    # Returns the title and artist of every submission from player
    song_list = submissions_df.loc[submissions_df['Submitter ID'] == player, ['Title','Artist(s)']]
    # smush the artist and title together
    song_list['formatted'] = song_list['Title'].str.cat(song_list['Artist(s)'], sep=' by ')
    # turn player id to player name
    player_name = get_player_name_from_id(player)  
    # drop the ide field
    song_list = song_list.iloc[:, 1:]
    # dict of list of songs index by player name
    songs_submitted[player_name] = song_list['formatted'].tolist()
    
#sort the dict alphabetically to make it pretty
songs_submitted = dict(sorted(songs_submitted.items()))

# Define the event handler function for drop down
def on_change(change):
    if change['type'] == 'change' and change['name'] == 'value':
        with output:
            output.clear_output() # Clear previous text
            print(f"{BOLD}Player {change['new']} submitted the following songs:{RESET}")
            print(*songs_submitted[change['new']], sep="\n")
            
# Create the dropdown
dropdown = widgets.Dropdown(
    options=list(songs_submitted),
    description='Select a player:',
    style={'description_width': 'initial'},
    disabled=False,
)
output = widgets.Output()

# Bind the function to the dropdown's value changes
dropdown.observe(on_change)

# Display both the dropdown and the output container
display(dropdown, output)

Dropdown(description='Select a player:', options=('Adam Kestian', 'Ali N.', 'Andrew', 'Andy Curtis', 'Brigette…

Output()

In [3]:
# Calculate player stats
player_stats = []
for player in playerIds:
    # Sum all votes given to a player
    totalVotes = votes_df.loc[votes_df['Submitter ID'] == player, 'Points Assigned'].sum()
    # find the number of rounds they participated in
    count = (submissions_df['Submitter ID'] == player).sum()
    #print (player +' got '+ str(totalVotes) +' votes '+' and participated in '+str(count)+'rounds')
    # Append simple dictionaries to a list
    player_stats.append({'Name': get_player_name_from_id(player), 'Votes For': totalVotes, 'Rounds Participated':count})
    
# Convert the whole list to a DataFrame all at once
player_stat_df = pd.DataFrame(player_stats)
player_stat_df['Votes per Round'] = player_stat_df['Votes For']/player_stat_df['Rounds Participated']


## Votes per Round


In [4]:
# Display list of best votes per round
# sort by votes/round
player_stat_df = player_stat_df.sort_values(by='Votes per Round', ascending=False)
display_df = player_stat_df[['Name', 'Votes per Round']]
# display list
display(display_df.style.hide(axis="index"))

Name,Votes per Round
jandrews,39.181818
Andrew,38.333333
De Wolfe,36.280000
Isaiah Freed,34.440000
mandeepkundal,32.960000
peter mccomb,32.833333
Katie,31.913043
KLee,31.578947
Adam Kestian,31.440000
William Dullemond,31.400000


In [5]:
# how many rounds did each player not submit or not vote in?
# Find rows where Department is 'Sales' AND Region is 'East'
# Equivalent to filtering down data before a SUMIFS or COUNTIFS
for player in playerIds:
    rounds_submitted = df.loc[(df['Department'] == 'Sales') & (df['Region'] == 'East')]
    rounds_voted = df.loc[(df['Department'] == 'Sales') & (df['Region'] == 'East')]
print(filtered_df)

In [6]:
# player stats (best round [placement or votes], worst round, who gave you most/least votes, rounds missed
# graph of votes received over round (with a running average line)

In [ ]:
# Round stats who submitted last, who voted last, most votes given by a single person